## Data Collection 

### Importing Data from Kaggle and creating relevant directories to hold it 

In [162]:
import kagglehub
from pathlib import Path
# DATA IS SOURCED FROM THESE TWO LINKS 
#https://www.kaggle.com/datasets/snehaanbhawal/resume-dataset
#https://www.kaggle.com/datasets/jatinchawda/job-titles-and-description/data

#Create directory to store resume data set 
path = Path("resume_dataset")
path.mkdir(exist_ok=True)
if(path.is_dir() and not any(path.iterdir())):
    resume_path = kagglehub.dataset_download("snehaanbhawal/resume-dataset", output_dir=path)

#Create path directory to store job description dataset
path = Path("jd_dataset")
path.mkdir(exist_ok=True)
if(path.is_dir() and not any(path.iterdir())):
    jd_path = kagglehub.dataset_download("jatinchawda/job-titles-and-description", output_dir=path)

## Data Evaluation 

### Data Cleaning

In [163]:
import pandas as pd
#View uncleaned dataset to see how data might need to be cleaned
uncleaned_resume = pd.read_csv(r"resume_dataset/Resume/Resume.csv")
uncleaned_resume.head()

,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR


In [164]:
#We see that the data is separated into ID, Resume Str, Resume HTML, and Category.
#As such, we care for the categorical data present in category, and for the actual text values in Resume_Str,
#Requiring us to clean the unneeded rows.
uncleaned_resume = uncleaned_resume.drop(columns=["ID", "Resume_html", "Category"])
uncleaned_resume.head()

,Resume_str
0,HR ADMINISTRATOR/MARKETING ASSOCIATE\...
1,"HR SPECIALIST, US HR OPERATIONS ..."
2,HR DIRECTOR Summary Over 2...
3,HR SPECIALIST Summary Dedica...
4,HR MANAGER Skill Highlights ...


In [165]:
#Cleaning up unnecessary spaces in resumes
print(uncleaned_resume.loc[0, "Resume_str"])
uncleaned_resume["Resume_str"] = uncleaned_resume["Resume_str"].replace(to_replace="\n+", value = "\n", regex =True)
uncleaned_resume["Resume_str"] = uncleaned_resume["Resume_str"].replace(to_replace="[ \t]+", value = " ", regex =True)
print("")
print(uncleaned_resume.loc[0, "Resume_str"])
cleaned_resume = uncleaned_resume


         HR ADMINISTRATOR/MARKETING ASSOCIATE

HR ADMINISTRATOR       Summary     Dedicated Customer Service Manager with 15+ years of experience in Hospitality and Customer Service Management.   Respected builder and leader of customer-focused teams; strives to instill a shared, enthusiastic commitment to customer service.         Highlights         Focused on customer satisfaction  Team management  Marketing savvy  Conflict resolution techniques     Training and development  Skilled multi-tasker  Client relations specialist           Accomplishments      Missouri DOT Supervisor Training Certification  Certified by IHG in Customer Loyalty and Marketing by Segment   Hilton Worldwide General Manager Training Certification  Accomplished Trainer for cross server hospitality systems such as    Hilton OnQ  ,   Micros    Opera PMS   , Fidelio    OPERA    Reservation System (ORS) ,   Holidex    Completed courses and seminars in customer service, sales strategies, inventory control, loss preve

In [166]:
#File size of JD is too large to run consistently, take a sample 
#to work with instead
file = Path("jd_dataset/jd.csv")
if not file.exists():
    #Grab an equivalent number of rows to resume data set for quicker
    #iteration going forwards
    uncleaned_jd = pd.read_parquet(r"jd_dataset\clean_data.parquet")
    sample = uncleaned_jd.sample(2482, random_state=1)
    sample.to_csv("jd_dataset/jd.csv", index=False)
cleaned_jd = pd.read_csv(r"jd_dataset\jd.csv")
cleaned_jd.head()

,job_title,job_desc
0,Technical Conveyancer,The Firm: -Highly recognised firm based in Lee...
1,Food Production/Bakery Op,THE COMPANY Westray Recruitment Consultants ar...
2,"Associate, Enhanced Due Diligence","Job Summary The Associate, Enhanced Due Dilige..."
3,"Lead Technical Consultant - up to £80,000 + Be...","Lead Technical Consultant - up to £80,000 + Be..."
4,SMX - Air Traffic Ctrl Spec Terminal,Position Description: Are you an air traffic c...


In [167]:
cleaned_resume


,Resume_str
0,HR ADMINISTRATOR/MARKETING ASSOCIATE\nHR ADMI...
1,"HR SPECIALIST, US HR OPERATIONS Summary Versa..."
2,HR DIRECTOR Summary Over 20 years experience ...
3,"HR SPECIALIST Summary Dedicated, Driven, and ..."
4,HR MANAGER Skill Highlights HR SKILLS HR Depa...
...,...
2479,RANK: SGT/E-5 NON- COMMISSIONED OFFICER IN CH...
2480,"GOVERNMENT RELATIONS, COMMUNICATIONS AND ORGA..."
2481,GEEK SQUAD AGENT Professional Profile IT supp...
2482,PROGRAM DIRECTOR / OFFICE MANAGER Summary Hig...


## Feature Engineering

In [168]:
#To better compare the datasets, we add a title column to the resume dataset for future use
cleaned_resume["resume_title"] = cleaned_resume["Resume_str"].str.extract(r"^([0-9A-ZÉ;:'/&.,\- ]+)\s")
#Found Specific rows that did not include data or specifical
print(cleaned_resume[cleaned_resume["resume_title"].isna()].shape)
print(cleaned_resume[cleaned_resume["resume_title"].isna()])
cleaned_resume = cleaned_resume.dropna()




(24, 2)
                                             Resume_str resume_title
61     Summary Chicago\nHR generalist offering \n Re...          NaN
193    Zachory Edmiston Summary Skilled in Customer ...          NaN
222    Christopher Townes Summary Knowledgeable Info...          NaN
249    Objective To obtain a position in the informa...          NaN
416    Kimberly Fisheli Summary Dedicated and respon...          NaN
419    Marilyn Hunter Summary Focused on providing p...          NaN
420    Kpandipou Koffi Summary Compassionate teachin...          NaN
474    Summary Administrative support professional w...          NaN
656                                                              NaN
691    Rachael Lobdell Summary . Compassionate Senio...          NaN
949    PROJECT(S) MANAGER Professional Overview A Me...          NaN
1080   Camryn Hilliard Professional Summary Highly m...          NaN
1157   Qualifications Microsoft Office Specialist, S...          NaN
1218   Pavithra Shetty Sum

In [169]:
#Verify all columns are not empty in each data set
print(cleaned_resume["resume_title"].isna().sum())
print(cleaned_resume["Resume_str"].isna().sum())
print(cleaned_jd["job_title"].isna().sum())
print(cleaned_jd["job_desc"].isna().sum())

0
0
0
0


In [170]:
#We then create an overall paired data set of each resume and job description, allowing us to compare each through
#a variety of different metrics per split 
print(cleaned_resume["resume_title"].isna().sum())
resume_sample = cleaned_resume.sample(500, random_state=1)
jd_sample = cleaned_jd.sample(500, random_state=1)
comparison_df = resume_sample.merge(jd_sample, how="cross")
print(comparison_df.shape)
comparison_df.head()


0
(250000, 4)


,Resume_str,resume_title,job_title,job_desc
0,DIRECTOR OF DIGITAL INNOVATION AND STRATEGY E...,DIRECTOR OF DIGITAL INNOVATION AND STRATEGY,Recruitment Officer,A fantastic opportunity has arisen for a Recru...
1,DIRECTOR OF DIGITAL INNOVATION AND STRATEGY E...,DIRECTOR OF DIGITAL INNOVATION AND STRATEGY,Clinical Pharmacy Manager Adult Heart Transplant,"LocationNew York, New YorkShift:Day Flex (Unit..."
2,DIRECTOR OF DIGITAL INNOVATION AND STRATEGY E...,DIRECTOR OF DIGITAL INNOVATION AND STRATEGY,Service Sales Engineer,Assists his superior in ensuring adherence to ...
3,DIRECTOR OF DIGITAL INNOVATION AND STRATEGY E...,DIRECTOR OF DIGITAL INNOVATION AND STRATEGY,Strategic Sales Executive,It's fun to work in a company where people tru...
4,DIRECTOR OF DIGITAL INNOVATION AND STRATEGY E...,DIRECTOR OF DIGITAL INNOVATION AND STRATEGY,General Hand,We are looking for experienced General Hands t...


In [ ]:
#Feature extraction from pairs

#Feature 1, Title similarities
import torch 
from sentence_transformers import SentenceTransformer
import numpy as np

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

#Generate embeddings and save them to prevent future issues
resume_path = Path("resume_title_embeddings.npy")
jd_path = Path("jd_title_embeddings.npy")
#Conditional to prevent recreating the embeddings each time
if not resume_path.exists() or not jd_path.exists():
    resume_title_embeddings = model.encode(comparison_df["resume_title"].to_list(), batch_size=12000, show_progress_bar=True, 
                                           convert_to_numpy=True, normalize_embeddings=True)
    np.save('resume_title_embeddings.npy', resume_title_embeddings)
    jd_title_embeddings = model.encode(comparison_df["job_title"].to_list(), batch_size=12000, show_progress_bar=True, 
                                       convert_to_numpy=True, normalize_embeddings=True)
    np.save('jd_title_embeddings.npy', jd_title_embeddings)

resume_title_embeddings = np.load('resume_title_embeddings.npy')
jd_title_embeddings = np.load('jd_title_embeddings.npy')

# As the vector embeddings are normalized, we simply calculate the dot product  
title_cosine_similarity = np.sum(resume_title_embeddings * jd_title_embeddings, axis=1)

#Consine_similarity from scikit learn calcualtes an NxN matrix, so all resumes are compared against jobs, 
#The diagonal contains the proper row to row comparison for each pair

#Begin constructing the features dataframe for the model
feature_df = pd.DataFrame(title_cosine_similarity, columns=["title_cosine_similarity"])
feature_df["title_cosine_similarity"].describe()

FileNotFoundError: [Errno 2] No such file or directory: 'jd_title_embeddings'

## Model Selection and Training

## Model Tuning